In [1]:
# Set workspace
import sys
import os
workspace_root = os.path.abspath(os.path.join(os.getcwd(), '..'))
if workspace_root not in sys.path:
    sys.path.insert(0, workspace_root)

# Libraries

In [2]:
# Libraries
import pandas as pd
import numpy as np

In [3]:
# .env
from dotenv import load_dotenv
import os

load_dotenv()
utmb_website_login_username = os.getenv("utmb_website_login_username")
utmb_website_login_password = os.getenv("utmb_website_login_password")

from helper_functions.utmb_api import (
    get_utmb_live_access_token,
    get_utmb_access_token,
    get_race_results,
)

utmb_live_access_token = get_utmb_live_access_token(
    utmb_username=utmb_website_login_username, 
    utmb_password=utmb_website_login_password
)

In [15]:
import requests
import pandas as pd


def get_runner_sectors(
    race_id: str,
    race_year: int,
    runner_id: int,
    printouts: bool = False
) -> pd.DataFrame:

    if printouts:
        print(f"Retrieving sector data for runner: {runner_id} ~ {race_id} ~ {race_year}")

    headers = {
        "Accept": "*/*",
        "User-Agent": "Mozilla/5.0",
        "Origin": "https://live.utmb.world",
        "Referer": "https://live.utmb.world/",
    }

    url = f"https://utmblive-api.utmb.world/runners/{runner_id}"

    res = requests.get(url, headers=headers, params={"locale": "sl"})
    res.raise_for_status()

    data = res.json()

    passing = data.get("detail", {}).get("passing", [])

    if not passing:
        if printouts:
            print("No sector data found")
        return pd.DataFrame()

    df = pd.json_normalize(passing)

    df = df.rename(columns={
        "pointId": "checkpoint_id",
        "datetimeIn": "arrival_time",
        "datetimeOut": "departure_time",
        "restTimeSeconds": "rest_time_seconds"
    })

    df["runner_id"] = runner_id
    df["race_id"] = race_id
    df["race_year"] = race_year

    df["arrival_time"] = pd.to_datetime(df["arrival_time"], errors="coerce")
    df["departure_time"] = pd.to_datetime(df["departure_time"], errors="coerce")

    return df

In [16]:
df = get_runner_sectors(
    race_id="utmb",
    race_year=2025,
    runner_id=10,
    printouts=True
)

Retrieving sector data for runner: 10 ~ utmb ~ 2025


HTTPError: 404 Client Error: Not Found for url: https://utmblive-api.utmb.world/runners/10?locale=sl

In [17]:
import requests

url = "https://utmblive-api.utmb.world/races/utmb/progressive"

data = requests.get(url, params={
    "type": "FINAL_RANKING",
    "page": 0,
    "limit": 50
}).json()

runner = data["runners"][0]
print(runner.keys())

KeyError: 'runners'

In [21]:
import requests
import pandas as pd


def get_runner_sectors(
    race_id: str,
    race_year: int,
    runner_id: int,
    access_token: str = None,
    printouts: bool = False
) -> pd.DataFrame:

    if printouts:
        print(f"Fetching runner sectors: {runner_id} ~ {race_id} ~ {race_year}")

    headers = {
        "Accept": "*/*",
        "User-Agent": "Mozilla/5.0",
        "Origin": "https://live.utmb.world",
        "Referer": "https://live.utmb.world/",
        "X-Tenant": f"{race_id}_{race_year}",   # 🔥 REQUIRED
    }

    if access_token:
        headers["Authorization"] = f"Bearer {access_token}"

    url = f"https://utmblive-api.utmb.world/runners/{runner_id}"

    res = requests.get(url, headers=headers, params={"locale": "sl"})
    res.raise_for_status()

    data = res.json()

    passing = data.get("detail", {}).get("passing", [])

    if not passing:
        if printouts:
            print("No sector data found")
        return pd.DataFrame()

    df = pd.json_normalize(passing)

    df = df.rename(columns={
        "pointId": "checkpoint_id",
        "datetimeIn": "arrival_time",
        "datetimeOut": "departure_time",
        "restTimeSeconds": "rest_time_seconds"
    })

    df["runner_id"] = runner_id
    df["race_id"] = race_id
    df["race_year"] = race_year

    df["arrival_time"] = pd.to_datetime(df["arrival_time"], errors="coerce")
    df["departure_time"] = pd.to_datetime(df["departure_time"], errors="coerce")

    return df

In [22]:
df = get_runner_sectors(
    race_id="utmb",
    race_year=2025,
    runner_id=10,
    printouts=True
)

Fetching runner sectors: 10 ~ utmb ~ 2025
No sector data found


In [24]:
race_id="utmb"
race_year=2025
runner_id=10
printouts=True

In [ ]:
headers = {
    "Accept": "*/*",
    "User-Agent": "Mozilla/5.0",
    "Origin": "https://live.utmb.world",
    "Referer": "https://live.utmb.world/",
    "X-Tenant": f"{race_id}_{race_year}",   
}

url = f"https://utmblive-api.utmb.world/runners/{runner_id}"

res = requests.get(url, headers=headers, params={"locale": "sl"})
res.raise_for_status()

data = res.json()

In [26]:
data

{'resume': {'bib': 10,
  'info': {'age': 34,
   'fullname': 'Tom EVANS',
   'initials': 'TE',
   'photo': 'worldseries/Members/8ec05363-1a0f-4bb1-be25-e3ac34f7d0b5',
   'photoWebTv': 'worldseries/Members/Admin/Tom_EVANS_md7xkc',
   'url': 'https://utmb.world/runner/1410311.tom.evans',
   'index': 910,
   'countryCode': 'GB',
   'category': '20-34M',
   'sex': 'H',
   'club': 'ASICS',
   'teamMembers': None},
  'ranking': {'scratch': 1, 'sex': 1, 'category': 1},
  'lastLocation': None,
  'start': '2025-08-29T15:45:00.000Z',
  'raceTime': '19:18:58',
  'raceId': 'utmb',
  'raceName': 'UTMB®',
  'raceCategory': '100m',
  'prediction': {'lastPointId': 144,
   'lastPassing': '2025-08-30T11:03:58.000Z',
   'nextPointId': None,
   'nextPointPrediction': None,
   'finishPrediction': None},
  'isFinisher': True,
  'status': 'f',
  'diffToFirst': '00:00:00'},
 'detail': {'passings': [{'pointId': 0,
    'datetimeIn': None,
    'datetimeOut': '2025-08-29T15:45:00.000Z',
    'restTimeSeconds': None